In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import anndata as ad
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.sparse import issparse

In [ ]:
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)

In [ ]:
# all regular visium samples analyzed at 55um resolution.
SAMPLE_NAME = "P22"
SAMPLE_DIR  = "sample_P22"

In [ ]:
adata = sc.read_visium(f"{ACC_DATA_ROOT}/spatial/reg_visium/{SAMPLE_DIR}", load_images=True)

In [ ]:
adata

In [ ]:
counter_MT= 0
for i in list(adata.var_names):
    if i.startswith("MT-"):
        counter_MT+=1
print('MT counter:', counter_MT)

counter_ribo = 0
for i in list(adata.var_names):
    if i.startswith("RPS") or i.startswith("RPL"):
        counter_ribo+=1
print('Ribo counter:', counter_ribo)

counter_hb = 0
for i in list(adata.var_names):
    if i.startswith("HB"):
        counter_hb+=1

print('HB counter:', counter_hb)


In [ ]:
adata.var["mt"]   = adata.var_names.str.startswith(("MT-", "mt-"))
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL", "rps", "rpl"))
adata.var["hb"]   = adata.var_names.str.match(r"^(HB[^(P)]|hb[^(p)])")

sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"],
    percent_top=[50, 100, 200], log1p=True, inplace=True,
)

In [ ]:
adata

In [ ]:
adata.var_names_make_unique()

In [ ]:
# --- QC plots ---
fig, axes = plt.subplots(1, 3, figsize=(14, 8))
# fig.subplots_adjust(hspace=0.5, wspace=0.4)  # increase vertical and horizontal gaps
fig.suptitle(f"QC — {SAMPLE_NAME}", fontsize=14)

# Row 1: histograms
for ax, col, color in [
    (axes[0], "total_counts",      "steelblue"),
    (axes[1], "n_genes_by_counts", "darkorange"),
    (axes[2], "pct_counts_mt",     "seagreen"),
]:
    ax.grid(False)
    ax.hist(adata.obs[col], bins=100, color=color, edgecolor="none")

    ax.set(xlabel=col, ylabel="Number of Bins")
    # if col == "total_counts":
    #     ax.set_xscale("log")
plt.tight_layout()
plt.savefig(f'{ACC_DATA_ROOT}/spatial/reg_visium/{SAMPLE_DIR}/output/images/qc_stats.png', 
            format='png',
            dpi = 600,
            bbox_inches = 'tight')
plt.show()
# plt.close()

In [ ]:
# no need to plot this as its just to visualize the

# # --- QC plots (larger panels, no loop, no ticks, compact colorbars) ---
# fig, axes = plt.subplots(1, 3, figsize=(24, 8))
# fig.subplots_adjust(wspace=0.35)
# fig.suptitle(f"QC — {SAMPLE_NAME}", fontsize=16)

# # Panel 1
# ax0 = axes[0]
# sc0 = ax0.scatter(
#     adata.obsm["spatial"][:, 0], adata.obsm["spatial"][:, 1],
#     c=adata.obs["log1p_total_counts"], s=15.0, cmap="magma", rasterized=True
# )
# ax0.set_title("log1p_total_counts")
# ax0.set_aspect("equal")
# ax0.invert_yaxis()
# ax0.grid(False)
# ax0.set_xticks([]); ax0.set_yticks([])
# ax0.tick_params(axis="both", which="both", bottom=False, top=False, left=False, right=False,
#                 labelbottom=False, labelleft=False)
# for side in ["top", "right", "left", "bottom"]:
#     ax0.spines[side].set_visible(False)
# # cbar0 = fig.colorbar(sc0, ax=ax0, shrink=0.72, fraction=0.035, pad=0.02, aspect=30)
# cbar0 = fig.colorbar(sc0, ax=ax0, shrink=1.0, fraction=0.035, pad=0.02, aspect=30)
# cbar0.ax.tick_params(labelsize=14)

# # Panel 2
# ax1 = axes[1]
# sc1 = ax1.scatter(
#     adata.obsm["spatial"][:, 0], adata.obsm["spatial"][:, 1],
#     c=adata.obs["log1p_n_genes_by_counts"], s=15.0, cmap="magma", rasterized=True
# )
# ax1.set_title("log1p_n_genes_by_counts")
# ax1.set_aspect("equal")
# ax1.invert_yaxis()
# ax1.grid(False)
# ax1.set_xticks([]); ax1.set_yticks([])
# ax1.tick_params(axis="both", which="both", bottom=False, top=False, left=False, right=False,
#                 labelbottom=False, labelleft=False)
# for side in ["top", "right", "left", "bottom"]:
#     ax1.spines[side].set_visible(False)
# # cbar1 = fig.colorbar(sc1, ax=ax1, shrink=0.72, fraction=0.035, pad=0.02, aspect=30)
# cbar1 = fig.colorbar(sc1, ax=ax1, shrink=1.0, fraction=0.035, pad=0.02, aspect=30)
# cbar1.ax.tick_params(labelsize=14)

# # Panel 3
# ax2 = axes[2]
# sc2 = ax2.scatter(
#     adata.obsm["spatial"][:, 0], adata.obsm["spatial"][:, 1],
#     c=adata.obs["pct_counts_mt"], s=15.0, cmap="Reds", rasterized=True
# )
# ax2.set_title("pct_counts_mt")
# ax2.set_aspect("equal")
# ax2.invert_yaxis()
# ax2.grid(False)
# ax2.set_xticks([]); ax2.set_yticks([])
# ax2.tick_params(axis="both", which="both", bottom=False, top=False, left=False, right=False,
#                 labelbottom=False, labelleft=False)
# for side in ["top", "right", "left", "bottom"]:
#     ax2.spines[side].set_visible(False)
# # cbar2 = fig.colorbar(sc2, ax=ax2, shrink=0.72, fraction=0.035, pad=0.02, aspect=30)
# cbar2 = fig.colorbar(sc2, ax=ax2, shrink=1.0, fraction=0.035, pad=0.02, aspect=30)
# cbar2.ax.tick_params(labelsize=14)

# plt.tight_layout()
# # plt.savefig(f'{ACC_DATA_ROOT}/spatial/sample_P17/outputs/images/on_tissue_qc_stats.png', 
# #             format='png',
# #             dpi = 600,
# #             bbox_inches = 'tight')
# plt.show()
# # plt.close()


In [ ]:
adata

In [ ]:
# QC thresholds (inspect QC plots, then adjust)
MIN_COUNTS = 100
MAX_COUNTS = 22000
MIN_BINS_PER_GENE = 50

# constant
MAX_MITO_PCT = 10

In [ ]:
# --- Filter ---
print(f"Pre-QC:  {adata.n_obs:,} bins × {adata.n_vars:,} genes")

sc.pp.filter_cells(adata, min_counts=MIN_COUNTS)
# sc.pp.filter_cells(adata, min_genes=MIN_GENES)
adata = adata[adata.obs["total_counts"] < MAX_COUNTS].copy()
adata = adata[adata.obs["pct_counts_mt"] < MAX_MITO_PCT].copy()
sc.pp.filter_genes(adata, min_cells=MIN_BINS_PER_GENE)

print(f"Post-QC: {adata.n_obs:,} bins × {adata.n_vars:,} genes")

In [ ]:
set(list(adata.obs['in_tissue']))

In [ ]:
# add site / sample info / outcome
adata.obs['site'] = adata.obs['in_tissue'].map({1: 'Floor of Mouth'})
adata.obs['batch'] = adata.obs['in_tissue'].map({1: 'P22'})
adata.obs['outcome'] = adata.obs['in_tissue'].map({1: 'Poor'})

In [ ]:
adata.obs

In [ ]:
def fix_arrow_strings(df):
    """Convert ArrowStringArray to object dtype for h5ad compatibility."""
    df = df.copy()
    
    # Fix index
    if isinstance(df.index.array, pd.arrays.ArrowStringArray):
        df.index = df.index.astype(object)
    
    # Fix columns
    for col in df.columns:
        series = df[col]
        if isinstance(series.array, pd.arrays.ArrowStringArray):
            # Plain Arrow-backed string column
            df[col] = series.astype(object)
        elif hasattr(series, 'cat'):
            # Categorical column — fix Arrow-backed categories
            cats = series.cat.categories
            if isinstance(cats.array, pd.arrays.ArrowStringArray):
                df[col] = series.cat.rename_categories(cats.astype(object))

    return df

adata.obs = fix_arrow_strings(adata.obs)
adata.var = fix_arrow_strings(adata.var)


adata.write_h5ad(f"{ACC_DATA_ROOT}/spatial/reg_visium/{SAMPLE_DIR}/output/filtered/{SAMPLE_NAME}_filtered.h5ad", compression = 'lzf')